# Part 6 (실습) — 체인 vs 에이전트 (중급 시작)

> **이 노트북의 목표**
> 같은 요구를 체인과 에이전트로 각각 처리해 동작 차이를 눈으로 봄. `create_agent`로 첫 에이전트를 만들고 "생각→행동→관찰" 루프가 실제로 도는 과정을 관찰함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~5
> ⚠️ 도구는 맛보기 수준임. 본격적 도구 정의는 Part 7.

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-3-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0)
print("준비 완료")

## 1. 체인으로는 안 되는 일 (Part 5 복습)

체인은 정해진 길만 감. "계산"이 필요한 질문을 줘도, 모델은 도구가 없어 스스로 계산을 시늉할 뿐임(틀릴 수 있음).

In [ ]:
# ChatPromptTemplate: 채팅 모델용 메시지 템플릿을 만드는 클래스
#   → .from_messages([(역할, 템플릿), ...]): 역할별 메시지 리스트 생성
#   → 역할: "system"(시스템), "human"(사용자), "ai"(모델 응답)
from langchain_core.prompts import ChatPromptTemplate
# StrOutputParser: AIMessage 객체에서 .content(텍스트)만 꺼내주는 파서
#   → 체인의 마지막에 붙여서 순수 문자열 결과를 얻을 때 사용
#   → 사용법: prompt | llm | StrOutputParser()
from langchain_core.output_parsers import StrOutputParser

# ─── LCEL 파이프(|) 체인 조립 ───
# | (파이프): 앞 부품의 출력을 뒤 부품의 입력으로 자동 연결
#   → 프롬프트 | 모델 | 파서 형태가 가장 기본적인 체인 구조
chain = ChatPromptTemplate.from_template("{q}") | model | StrOutputParser()

# 큰 수 곱셈 — 모델이 암산하면 틀리기 쉬움
print(chain.invoke({"q": "47281 × 39472 는 정확히 얼마야? 숫자만 답해."}))
print("→ 체인은 '계산 도구'를 쓸 줄 모름. 모델의 암산에 의존함(부정확 위험).")

## 2. 첫 에이전트 — create_agent

에이전트에 **계산 도구**를 주면, 모델이 "이건 계산 도구를 써야겠다"고 스스로 판단해 호출함.

### 문법
- `from langchain.agents import create_agent`
- `create_agent(model, tools=[...])` — 모델 + 도구 리스트
- `@tool` 데코레이터로 일반 함수를 도구로 만듦 (Part 7에서 자세히)

In [ ]:
from langchain.agents import create_agent
# @tool 데코레이터: 일반 파이썬 함수를 LangChain "도구"로 변환
#   → 함수의 docstring이 도구 설명이 되고, 파라미터가 도구 입력 스키마가 됨
#   → 에이전트가 이 설명을 읽고 언제 이 도구를 쓸지 스스로 판단함
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """수식 문자열을 계산한다. 예: '47281 * 39472'"""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"계산 오류: {e}"

agent = create_agent(model, tools=[calculator])
print("에이전트 생성 완료 (도구: calculator)")

In [ ]:
# 같은 질문을 에이전트에게 — 이번엔 도구를 써서 정확히 계산함
result = agent.invoke({"messages": [("user", "47281 곱하기 39472 는 얼마야?")]})

# 최종 답변은 messages의 마지막 메시지
print(result["messages"][-1].content)

## 3. 루프 관찰 — 생각 → 행동 → 관찰

에이전트의 `.invoke()`는 내부에서 모델을 여러 번 부를 수 있음. messages 전체를 펼쳐 보면 ReAct 루프가 보임.

In [ ]:
# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
result = agent.invoke({"messages": [("user", "(123 + 877) 를 5로 나누면?")]})

# messages 전체를 순서대로 출력 — 루프의 흔적
for m in result["messages"]:
    role = type(m).__name__
    # 도구 호출 정보가 있으면 표시
    if hasattr(m, "tool_calls") and m.tool_calls:
        for tc in m.tool_calls:
            print(f"[{role}] 🔧 도구 호출: {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"[{role}] 👁 관찰(도구 결과): {m.content}")
    elif m.content:
        print(f"[{role}] 💬 {m.content}")
    print("-"*50)

> 📌 위 출력에서 보이는 흐름:
> 1. **HumanMessage**: 사용자 질문
> 2. **AIMessage (tool_calls)**: 모델이 "계산 도구를 쓰자"고 판단 → 행동
> 3. **ToolMessage**: 도구 실행 결과 → 관찰
> 4. **AIMessage**: 관찰을 바탕으로 최종 답변
>
> 체인이라면 2~3단계가 없음. 이 루프가 에이전트의 본질임.

## 4. 에이전트는 도구를 '안 쓸' 수도 있다

도구가 필요 없는 질문이면, 에이전트는 호출 없이 바로 답함. **쓸지 말지도 모델의 판단**임.

In [ ]:
result = agent.invoke({"messages": [("user", "안녕? 너는 누구야?")]})

tool_used = any(
    getattr(m, "tool_calls", None) for m in result["messages"]
)
print("도구 사용 여부:", "사용함" if tool_used else "사용 안 함 (불필요)")
print("답변:", result["messages"][-1].content)

## 5. 나란히 비교 — 같은 질문, 다른 처리

| | 체인 | 에이전트 |
|---|---|---|
| 큰 수 계산 | 암산(부정확 위험) | 계산 도구 호출(정확) |
| 인사말 | 정해진 블록 통과 | 도구 없이 바로 답 |
| 흐름 결정 | 코드(고정) | 모델(판단) |
| 모델 호출 수 | 보통 1회 | 1~여러 회 |

In [ ]:
# 체인: 항상 같은 경로
print("[체인]", chain.invoke({"q": "5 더하기 7?"}))

# 에이전트: 판단해서 처리 (이 경우 간단해서 도구 없이 답할 수도)
r = agent.invoke({"messages": [("user", "5 더하기 7?")]})
print("[에이전트]", r["messages"][-1].content)

> 📌 **남용 경고**: 위 "5+7" 같은 단순 질문에 에이전트는 과함. 정해진 변환이면 체인이 빠르고 저렴함. 에이전트는 '동적 판단·도구·분기'가 필요할 때만.

## ✏️ 미니 실습

**과제**: 현재 시각을 알려주는 도구 `now()`를 만들고, 에이전트에 추가해 "지금 몇 시야?"라고 물어보기.
힌트: `from datetime import datetime`, `@tool` 데코레이터 사용.

In [ ]:
# TODO: 직접 작성
# from datetime import datetime
# @tool
# def now() -> str:
#     """현재 날짜와 시각을 돌려준다."""
#     return ...
# agent2 = create_agent(model, tools=[calculator, now])
# print(agent2.invoke({"messages": [("user", "지금 몇 시야?")]})["messages"][-1].content)

<details><summary>👉 정답</summary>

```python
from datetime import datetime

@tool
def now() -> str:
    """현재 날짜와 시각을 돌려준다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

agent2 = create_agent(model, tools=[calculator, now])
print(agent2.invoke({"messages": [("user", "지금 몇 시야?")]})["messages"][-1].content)
```
</details>

## 정리

### 3줄 요약
1. 체인은 정해진 길(코드가 흐름 결정), 에이전트는 모델이 행동을 판단함.
2. 에이전트는 생각→행동(도구)→관찰→반복 루프를 돌며, 도구를 안 쓸 수도 있음.
3. 에이전트는 동적 판단이 필요할 때만 — 단순 변환은 체인이 정석임.

### ▶ 다음 (Part 7)
에이전트의 손과 발인 **도구(Tool)**를 직접 정의하는 법. `@tool` 데코레이터, 함수 시그니처·docstring이 곧 도구 명세가 되는 원리를 배움.